In [ ]:
# H3 Video Studio -- PERSISTENT TEST WORKER (CPU, burns no GPU quota)
# One long session: polls the jobs dataset via the Kaggle API, renders ffmpeg
# placeholder clips for every new test job, pushes results to the outputs
# dataset after EVERY job, and exits after IDLE_TIMEOUT_S of no work.
# Real (non-test) jobs are marked needs_gpu and the worker exits so the site
# relaunches the GPU worker for them.
import json, os, shutil, subprocess, sys, time
from pathlib import Path

TOKEN = "KGAT_abba3bba83ed8f6a11855f7a819228ea"
JOBS_DS = "freedomform/rmc-studio-jobs"
OUTPUTS_DS = "freedomform/rmc-studio-outputs"
WORK = Path("/kaggle/working")
OUT = WORK / "out"
OUT.mkdir(parents=True, exist_ok=True)
POLL_S = 20
IDLE_TIMEOUT_S = 600
MAX_SESSION_S = int(11.9 * 3600)

os.environ["KAGGLE_API_TOKEN"] = TOKEN
STATE = {"jobs": {}}
SEEN = set()
PUSH_LOCK = __import__("threading").Lock()

def ensure_kaggle_cli():
    """the preinstalled 1.5.x CLI on Kaggle images cannot authenticate with a
    KAGGLE_API_TOKEN (KGAT_...) -- upgrade to the 1.6+/1.7.x line that can"""
    try:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"],
                           capture_output=True, text=True, timeout=600)
        print(f"[cli] kaggle upgrade rc={r.returncode}", flush=True)
    except Exception as exc:
        print(f"[cli] kaggle upgrade failed: {type(exc).__name__}: {exc}", flush=True)

ensure_kaggle_cli()

def kaggle(*args, timeout=900):
    return subprocess.run([sys.executable, "-m", "kaggle", *args],
                          capture_output=True, text=True, timeout=timeout)

def push_outputs(tag):
    try:
        with PUSH_LOCK:
            (OUT / "state.json").write_text(
                json.dumps(STATE, indent=1, ensure_ascii=False), encoding="utf-8")
            (OUT / "dataset-metadata.json").write_text(json.dumps(
                {"title": "rmc-studio-outputs", "id": OUTPUTS_DS,
                 "licenses": [{"name": "CC0-1.0"}]}), encoding="utf-8")
            created = kaggle("datasets", "create", "-p", str(OUT),
                             "--dir-mode", "skip", timeout=1800)
            if "successfully created" not in (created.stdout + created.stderr).lower():
                v = kaggle("datasets", "version", "-p", str(OUT),
                           "--dir-mode", "skip", "-m", tag, timeout=1800)
                print(f"outputs version rc={v.returncode} "
                      f"{(v.stdout + v.stderr)[-200:]}", flush=True)
            else:
                print("outputs dataset created (first push)", flush=True)
    except Exception as exc:
        print(f"outputs push failed (non-fatal): {type(exc).__name__}: {exc}",
              flush=True)

def jobs_line():
    try:
        r = kaggle("datasets", "list", "--mine", "--search", JOBS_DS, timeout=120)
        for line in r.stdout.splitlines():
            if line.startswith(JOBS_DS):
                return line
        print(f"[list] no match rc={r.returncode} out={r.stdout[:200]!r} "
              f"err={r.stderr[:300]!r}", flush=True)
    except Exception as exc:
        print(f"jobs_line failed: {type(exc).__name__}: {exc}", flush=True)
    return ""

def wait_ready(timeout_s=420):
    """tolerant gate: the preinstalled kaggle CLI on Kaggle images may print
    a different status string than our local 1.7.x, so treat anything that is
    not explicitly still-processing as ready and LOG the raw output for diagnosis"""
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        try:
            r = kaggle("datasets", "status", JOBS_DS, timeout=120)
            out = (r.stdout or "").strip().split("\n")[0].strip().lower()
            print(f"[status] rc={r.returncode} out={out[:60]!r} "
                  f"err={(r.stderr or '')[:80]!r}", flush=True)
            if "processing" not in out and "pending" not in out                     and "queued" not in out:
                return True
        except Exception as exc:
            print(f"[status] error {type(exc).__name__}: {exc}", flush=True)
        time.sleep(20)
    return False

def download_jobs(dest):
    shutil.rmtree(dest, ignore_errors=True)
    dest.mkdir(parents=True, exist_ok=True)
    try:
        kaggle("datasets", "download", JOBS_DS, "-p", str(dest), "--unzip")
    except Exception as exc:
        print(f"jobs download failed: {type(exc).__name__}: {exc}", flush=True)
        return None
    found = sorted(dest.glob("**/shots.json"))
    return found[0].parent if found else None

def render_stub(entry):
    jid = str(entry["job_id"])
    sec = int(entry["seconds"])
    dest = OUT / f"job_{jid}.mp4"
    cmd = ["ffmpeg", "-y", "-v", "error",
           "-f", "lavfi", "-i", f"color=c=0x1f2937:s=864x480:r=24:d={sec}",
           "-f", "lavfi", "-i",
           f"sine=frequency={330 + int(entry['num']) * 40}:duration={sec}",
           "-vf", ("drawtext=text='TEST {jid}':fontcolor=white@0.85:"
                   "x=(w-tw)/2:y=(h-th)/2:fontsize=28"),
           "-c:v", "libx264", "-preset", "veryfast", "-crf", "28",
           "-c:a", "aac", "-b:a", "96k", "-shortest", "-movflags", "+faststart",
           str(dest)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    assert r.returncode == 0, f"ffmpeg failed for job {jid}: {r.stderr[-500:]}"
    return dest

started = time.time()
last_work = time.time()
last_mark = ""
print("test worker online (CPU)", flush=True)
while time.time() - started < MAX_SESSION_S:
    if time.time() - last_work > IDLE_TIMEOUT_S:
        print("idle timeout -> exiting to save resources", flush=True)
        break
    mark = jobs_line()
    if not mark or mark == last_mark:
        time.sleep(POLL_S)
        continue
    if not wait_ready():
        continue
    base = download_jobs(WORK / "jobs_dl")
    if base is None:
        time.sleep(POLL_S)
        continue
    try:
        manifest = json.loads((base / "shots.json").read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"bad manifest: {type(exc).__name__}: {exc}", flush=True)
        time.sleep(POLL_S)
        continue
    done_ids = base / "done_ids.json"
    if done_ids.is_file():
        try:
            SEEN.update(json.loads(done_ids.read_text(encoding="utf-8")))
        except Exception:
            pass
    test_batch = (base / "test_mode.txt").is_file()
    fresh = [e for e in manifest if str(e.get("job_id")) not in SEEN]
    if not fresh:
        time.sleep(POLL_S)
        continue
    if not test_batch:
        for e in fresh:
            jid = str(e.get("job_id") or f"n{e.get('num')}")
            STATE["jobs"][jid] = {"status": "needs_gpu",
                                  "note": "CPU test worker cannot render real jobs"}
            print(f"real job {jid} arrived -> needs_gpu", flush=True)
        push_outputs("needs_gpu")
        print("real jobs present -> exiting so the site launches the GPU worker",
              flush=True)
        break
    for e in fresh:
        jid = str(e.get("job_id") or f"n{e.get('num')}")
        try:
            dest = render_stub(e)
            STATE["jobs"][jid] = {"status": "done", "file": str(dest),
                                  "bytes": dest.stat().st_size, "test_stub": True}
            SEEN.add(jid)
            last_work = time.time()
            push_outputs(f"stub {jid}")
            print(f"job {jid}: stub rendered + pushed", flush=True)
        except Exception as exc:
            STATE["jobs"][jid] = {"status": "failed",
                                  "error": f"{type(exc).__name__}: {exc}"[:600]}
            SEEN.add(jid)
            push_outputs(f"stub failed {jid}")
            print(f"job {jid}: stub FAILED: {exc}", flush=True)
    last_mark = mark
print(f"test worker finished after {(time.time() - started) / 60:.1f} min; "
      f"{sum(1 for v in STATE['jobs'].values() if v.get('status') == 'done')} done",
      flush=True)
